# Vietnamese–English Code-Switching Tagger
**Task:** Given a mixed Vietnamese-English sentence, label every word as **VI** (Vietnamese) or **EN** (English).

**Before running:** click the 📁 Files icon on the left and drag in `train_marked.txt` and `canvec_marked.txt` (made by `python src/canvec.py prelabel`, then checked by hand). Then run each cell top to bottom (Shift+Enter).

## 1. The tools
How it works in plain words:
- For each word, we describe it with **features**: its letter chunks (e.g. `ng`, `ươ`, `th` are very Vietnamese; `ck`, `sh`, `tion` are very English), whether it has a Vietnamese tone mark, and the words right before and after it.
- A **logistic regression** model learns from labeled examples which features point to VI and which point to EN.
- To tag a new sentence, it computes features for each word and picks the more likely label.

In [ ]:
import re, random, unicodedata
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline


def load_marked(path):
    """Read sentences where English words start with '*'. Returns list of (tokens, labels)."""
    data = []
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        toks, labs = [], []
        for w in line.split():
            labs.append("EN" if w.startswith("*") else "VI")
            toks.append(w.lstrip("*"))
        data.append((toks, labs))
    return data


def strip_accents(text):
    """Remove Vietnamese tone marks: 'Hôm nay' -> 'Hom nay'. Also đ -> d."""
    text = text.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn")


# Revision 2: the 100 most frequent English words (Oxford English Corpus list),
# plus forms of be/do/have. Chosen from a general list, NOT from test-set errors.
# Some collide with Vietnamese typed without accents (me=mẹ, an=ăn, do=đó, the=thể,
# no=nó, can=cần); for those the model has to rely on the neighbouring words.
COMMON_EN = set("""
the be to of and a in that have i it for not on with he as you do at this but his by
from they we say her she or an will my one all would there their what so up out if about
who get which go me when make can like time no just him know take people into year your
good some could them see other than then now look only come its over think also back after
use two how our work first well way even new want because any these give day most us
is are was were am been being has had did does i'm it's don't
""".split())


def word_features(tokens, i, use_lexicon=False):
    """Describe word i using its letters and its neighbours."""
    w = tokens[i].lower()
    padded = f"<{w}>"
    f = {"word=" + w: 1, "has_vn_mark": int(strip_accents(w) != w)}
    if use_lexicon:
        f["common_en"] = int(w in COMMON_EN)
        f["prev_common_en"] = int(i > 0 and tokens[i - 1].lower() in COMMON_EN)
        f["next_common_en"] = int(i < len(tokens) - 1 and tokens[i + 1].lower() in COMMON_EN)
    for n in (1, 2, 3):
        for j in range(len(padded) - n + 1):
            f[f"ng{n}=" + padded[j:j + n]] = 1
    f["prev=" + (tokens[i - 1].lower() if i > 0 else "<START>")] = 1
    f["next=" + (tokens[i + 1].lower() if i < len(tokens) - 1 else "<END>")] = 1
    return f


def to_xy(data, use_lexicon=False):
    X, y = [], []
    for toks, labs in data:
        for i in range(len(toks)):
            X.append(word_features(toks, i, use_lexicon))
            y.append(labs[i])
    return X, y


def train(data, use_lexicon=False):
    X, y = to_xy(data, use_lexicon)
    model = make_pipeline(DictVectorizer(), LogisticRegression(max_iter=2000, C=5))
    model.fit(X, y)
    model.use_lexicon = use_lexicon   # remember which features this model expects
    return model


def features(model, tokens):
    return [word_features(tokens, i, getattr(model, "use_lexicon", False)) for i in range(len(tokens))]


def tag(model, sentence):
    toks = re.findall(r"[\wÀ-ỹ']+", sentence)
    preds = model.predict(features(model, toks))
    return list(zip(toks, preds))


def accuracy(model, data):
    X, y = to_xy(data, getattr(model, "use_lexicon", False))
    p = model.predict(X)
    scored = [(x, a, b) for x, a, b in zip(X, y, p) if a != "OTHER"]   # names/fillers are not scored
    correct = sum(a == b for _, a, b in scored)
    return correct / len(scored), [(x, a, b) for x, a, b in scored if a != b]


def no_accent_copy(data):
    return [([strip_accents(t) for t in toks], labs) for toks, labs in data]


def load_canvec_marked(path):
    """Like load_marked, but words starting with ~ (names, fillers) get label OTHER and are not scored."""
    data = []
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        toks, labs = [], []
        for w in line.split():
            labs.append("EN" if w.startswith("*") else "OTHER" if w.startswith("~") else "VI")
            toks.append(w.lstrip("*~"))
        data.append((toks, labs))
    return data

## 2. Load the training data
In `train_marked.txt`, English words start with `*`; everything else is Vietnamese.

In [ ]:
train_data = load_marked("train_marked.txt")
print(len(train_data), "training sentences,", sum(len(t) for t, _ in train_data), "words")
n_en = sum(l == "EN" for _, labs in train_data for l in labs)
print("EN words:", n_en, "| VI words:", sum(len(t) for t, _ in train_data) - n_en)
print("Example:", list(zip(*train_data[0])))

## 3. Test sets (never used for training)
**Test set A:** CanVEC sample, 95 utterances of real Vietnamese-English family conversation (Nguyen & Bryant 2020). The public sample has **no word-level labels**. We pre-labelled it with simple rules (a word with a tone mark is VI; everything else is EN unless it's on a short list of unaccented Vietnamese words) and then **checked every line by hand**. Names (Jimmy, BTS…) and fillers (mhm, uh) are marked `~` and not scored. We also test a **no-accent copy** of CanVEC, because people often text without tone marks.

**Test set B:** your own chat-style sentences **without accents**, the challenging case. Write them in the same `*english` format. ⚠️ The three below are placeholders: replace them with your own ~20 sentences.

In [ ]:
# Test set A: CanVEC (hand-checked labels)
try:
    test_canvec = load_canvec_marked("canvec_marked.txt")
except FileNotFoundError:
    test_canvec = []
    print("canvec_marked.txt not found: upload it to run test set A")
test_canvec_noacc = no_accent_copy(test_canvec)
print(len(test_canvec), "CanVEC test sentences")

# Test set B: YOUR OWN no-accent sentences (replace these placeholders!)
my_sentences = '''
hom nay minh phai lam *homework toi khuya
toi qua *team minh *meeting mai moi xong
mai di an *brunch khong
'''
test_mine = []
for line in my_sentences.strip().splitlines():
    toks = line.split()
    test_mine.append(([w.lstrip("*") for w in toks], ["EN" if w.startswith("*") else "VI" for w in toks]))
print(len(test_mine), "of my own test sentences")

TEST_SETS = [("CanVEC", test_canvec), ("CanVEC no-accent", test_canvec_noacc), ("My no-accent sentences", test_mine)]


def report(models):
    """Print a table: one row per test set, one column per model."""
    print(f"{'Test set':26}" + "".join(f"{name:>11}" for name, _ in models))
    for ts_name, ts in TEST_SETS:
        if ts:
            print(f"{ts_name:26}" + "".join(f"{accuracy(m, ts)[0]:>11.1%}" for _, m in models))

## 4. Baseline model (trained on accented text only)

In [ ]:
baseline = train(train_data)
report([("Baseline", baseline)])

## 5. Demo: tag any sentence

In [ ]:
def show(model, sentence):
    print("  ".join(f"{w}/{lab}" for w, lab in tag(model, sentence)))

show(baseline, "Tối nay đi ăn không mình hơi bored")
show(baseline, "hom nay minh co deadline nen phai rush lam")  # no accents: watch for mistakes

## 6. Revision: add no-accent copies of the training data
**Observed problem:** the baseline makes many more mistakes when tone marks are missing (common in real texting).
**Possible cause:** it learned that tone marks ⇒ Vietnamese, so unaccented Vietnamese words like `hom`, `lam` look English.
**One change:** duplicate every training sentence with accents stripped, then retrain and rerun the *same* tests.

In [ ]:
revised = train(train_data + no_accent_copy(train_data))
report([("Baseline", baseline), ("Revised", revised)])
show(revised, "hom nay minh co deadline nen phai rush lam")

## 7. Revision 2: add a "common English word" feature
**Observed problem:** on real speech (CanVEC), revision 1 got *worse* on accented text. It started calling short English words like `a`, `it`, `that`, `but`, `to` Vietnamese.
**Possible cause:** the training sentences almost never contain English function words. Their English words are mostly nouns and verbs dropped into Vietnamese (`deadline`, `rush`), and only 10 of 155 sentences are mostly English. Once tone marks are stripped, a short word with no mark looks like it could be either language.
**One change:** add a feature saying whether the word (and its neighbours) is on a fixed list of the 100 most common English words (`COMMON_EN`, Oxford English Corpus list). The list came from a general source, **not** from our test errors, so the test stays fair.

In [ ]:
revised2 = train(train_data + no_accent_copy(train_data), use_lexicon=True)
report([("Baseline", baseline), ("Revised", revised), ("Revised 2", revised2)])
show(revised2, "hom nay minh co deadline nen phai rush lam")
show(revised2, "tại vì he is my ideal type")

## 8. Remaining errors (use these for your Limitations slide)

In [ ]:
for name, ts in TEST_SETS:
    if ts:
        _, errors = accuracy(revised2, ts)
        print(f"--- {name}: {len(errors)} errors (Revised 2)")
        for feats, true, pred in errors[:15]:
            word = [k[5:] for k in feats if k.startswith("word=")][0]
            print(f"  {word!r}: expected {true}, got {pred}")